In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

# 원본 데이터 루트
DATA_ROOT = Path(
    "/content/drive/MyDrive/빅프/"
    "081_스마트야드_압축해제/"
    "3.개방데이터/1.데이터"
)

# 원본을 건드리지 않고 분석 결과만 저장할 폴더
WORK_ROOT = Path(
    "/content/drive/MyDrive/빅프/"
    "shared_backbone_workspace"
)

REPORT_ROOT = WORK_ROOT / "reports"
MANIFEST_ROOT = WORK_ROOT / "manifests"
VIS_ROOT = WORK_ROOT / "visualizations"

for path in [REPORT_ROOT, MANIFEST_ROOT, VIS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("WORK_ROOT:", WORK_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())

assert DATA_ROOT.exists(), f"데이터 루트를 찾을 수 없습니다: {DATA_ROOT}"

DATA_ROOT: /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터
WORK_ROOT: /content/drive/MyDrive/빅프/shared_backbone_workspace
DATA_ROOT exists: True


In [3]:
def print_tree(root: Path, max_depth: int = 3):
    root = root.resolve()
    print(root.name)

    for path in sorted(root.rglob("*")):
        try:
            depth = len(path.relative_to(root).parts)
        except ValueError:
            continue

        if depth > max_depth:
            continue

        indent = "    " * depth
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}{path.name}{suffix}")


print_tree(DATA_ROOT, max_depth=3)

1.데이터
    Training/
        01.원천데이터/
            TS_안전장비-S63_DATA3_안전대미착용-B0/
            TS_안전장비-S63_DATA3_안전대미착용-B0.zip
            TS_안전장비-S63_DATA3_안전대착용-B1/
            TS_안전장비-S63_DATA3_안전대착용-B1.zip
            TS_안전장비-S63_DATA3_안전모미착용-H0/
            TS_안전장비-S63_DATA3_안전모미착용-H0.zip
            TS_안전장비-S63_DATA3_안전모착용-H1/
            TS_안전장비-S63_DATA3_안전모착용-H1.zip
            TS_안전장비-S63_DATA3_용접마스크미착용-M0/
            TS_안전장비-S63_DATA3_용접마스크미착용-M0.zip
            TS_안전장비-S63_DATA3_용접마스크착용-M1/
            TS_안전장비-S63_DATA3_용접마스크착용-M1.zip
        02.라벨링데이터/
            TL_안전장비-S63_DATA3_안전대미착용-B0/
            TL_안전장비-S63_DATA3_안전대미착용-B0.zip
            TL_안전장비-S63_DATA3_안전대착용-B1/
            TL_안전장비-S63_DATA

In [4]:
from collections import Counter, defaultdict
import pandas as pd

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
JSON_EXTENSION = ".json"


def collect_files(root: Path):
    image_paths = []
    json_paths = []

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        if suffix in IMAGE_EXTENSIONS:
            image_paths.append(path)
        elif suffix == JSON_EXTENSION:
            json_paths.append(path)

    return sorted(image_paths), sorted(json_paths)


image_paths, json_paths = collect_files(DATA_ROOT)

print(f"전체 이미지 수: {len(image_paths):,}")
print(f"전체 JSON 수: {len(json_paths):,}")

print("\n이미지 확장자:")
print(Counter(path.suffix.lower() for path in image_paths))

print("\n상위 JSON 폴더:")
for folder, count in Counter(str(p.parent) for p in json_paths).most_common(20):
    print(f"{count:>8,}  {folder}")

전체 이미지 수: 76,966
전체 JSON 수: 76,966

이미지 확장자:
Counter({'.jpg': 76966})

상위 JSON 폴더:
  13,632  /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_안전대미착용-B0
  13,559  /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_안전모착용-H1
  13,502  /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_안전모미착용-H0
  13,303  /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_안전대착용-B1
   7,480  /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_용접마스크미착용-M0
   6,938  /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_용접마스크착용-M1
   1,704  /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Validation/02.라벨링데이터

In [5]:
def build_unique_stem_map(paths):
    stem_map = defaultdict(list)

    for path in paths:
        stem_map[path.stem].append(path)

    return stem_map


image_stem_map = build_unique_stem_map(image_paths)
json_stem_map = build_unique_stem_map(json_paths)

duplicate_image_stems = {
    stem: paths
    for stem, paths in image_stem_map.items()
    if len(paths) > 1
}

duplicate_json_stems = {
    stem: paths
    for stem, paths in json_stem_map.items()
    if len(paths) > 1
}

image_stems = set(image_stem_map)
json_stems = set(json_stem_map)

images_without_json = sorted(image_stems - json_stems)
jsons_without_image = sorted(json_stems - image_stems)

print(f"중복 이미지 stem: {len(duplicate_image_stems):,}")
print(f"중복 JSON stem: {len(duplicate_json_stems):,}")
print(f"JSON 없는 이미지: {len(images_without_json):,}")
print(f"이미지 없는 JSON: {len(jsons_without_image):,}")

assert not duplicate_image_stems, "중복 이미지 stem이 있습니다."
assert not duplicate_json_stems, "중복 JSON stem이 있습니다."

중복 이미지 stem: 0
중복 JSON stem: 0
JSON 없는 이미지: 0
이미지 없는 JSON: 0


In [6]:
paired_records = []

for stem in sorted(image_stems & json_stems):
    image_path = image_stem_map[stem][0]
    json_path = json_stem_map[stem][0]

    paired_records.append({
        "stem": stem,
        "image_path": str(image_path),
        "json_path": str(json_path),
    })

pairs_df = pd.DataFrame(paired_records)

print("매칭 수:", len(pairs_df))
display(pairs_df.head())

매칭 수: 76966


,stem,image_path,json_path
0,S63_DATA3_B0_L1_D2023-08-25-10-50_001_000254,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
1,S63_DATA3_B0_L1_D2023-08-25-10-50_001_000255,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
2,S63_DATA3_B0_L1_D2023-08-25-10-50_001_000256,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
3,S63_DATA3_B0_L1_D2023-08-25-10-50_001_000257,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
4,S63_DATA3_B0_L1_D2023-08-25-10-50_001_000258,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...


In [7]:
import re

KNOWN_PPE_CODES = {"B0", "B1", "H0", "H1", "M0", "M1"}

# "_B0_", "_H1_" 같은 패턴
PPE_CODE_PATTERN = re.compile(r"_(B0|B1|H0|H1|M0|M1)_")


def extract_ppe_code(filename_stem: str):
    match = PPE_CODE_PATTERN.search(filename_stem)
    return match.group(1) if match else None


pairs_df["filename_ppe_code"] = pairs_df["stem"].map(extract_ppe_code)

print("파일명 PPE 코드 통계:")
display(
    pairs_df["filename_ppe_code"]
    .value_counts(dropna=False)
    .rename_axis("code")
    .reset_index(name="count")
)

failed_code_df = pairs_df[pairs_df["filename_ppe_code"].isna()]

print("코드 판별 실패:", len(failed_code_df))
display(failed_code_df.head())

파일명 PPE 코드 통계:


,code,count
0,B0,15336
1,H1,15254
2,H0,15190
3,B1,14966
4,M0,8415
5,M1,7805


코드 판별 실패: 0


,stem,image_path,json_path,filename_ppe_code


In [8]:
import json
from typing import Any


def load_json(json_path: Path):
    with json_path.open("r", encoding="utf-8-sig") as f:
        return json.load(f)


def flatten_json_structure(
    obj: Any,
    prefix: str = "root",
    max_list_items: int = 3,
    max_depth: int = 8,
    current_depth: int = 0,
):
    rows = []

    if current_depth > max_depth:
        rows.append({
            "path": prefix,
            "type": "MAX_DEPTH_REACHED",
            "example": None,
        })
        return rows

    if isinstance(obj, dict):
        rows.append({
            "path": prefix,
            "type": "dict",
            "example": f"{len(obj)} keys",
        })

        for key, value in obj.items():
            child_prefix = f"{prefix}.{key}"
            rows.extend(
                flatten_json_structure(
                    value,
                    prefix=child_prefix,
                    max_list_items=max_list_items,
                    max_depth=max_depth,
                    current_depth=current_depth + 1,
                )
            )

    elif isinstance(obj, list):
        rows.append({
            "path": prefix,
            "type": "list",
            "example": f"length={len(obj)}",
        })

        for index, value in enumerate(obj[:max_list_items]):
            child_prefix = f"{prefix}[{index}]"
            rows.extend(
                flatten_json_structure(
                    value,
                    prefix=child_prefix,
                    max_list_items=max_list_items,
                    max_depth=max_depth,
                    current_depth=current_depth + 1,
                )
            )

    else:
        example = obj

        if isinstance(example, str) and len(example) > 150:
            example = example[:150] + "..."

        rows.append({
            "path": prefix,
            "type": type(obj).__name__,
            "example": example,
        })

    return rows

In [9]:
representative_rows = []

for code in sorted(KNOWN_PPE_CODES):
    code_rows = pairs_df[pairs_df["filename_ppe_code"] == code]

    if code_rows.empty:
        print(f"[경고] {code} 파일이 없습니다.")
        continue

    selected = code_rows.iloc[0]

    representative_rows.append({
        "code": code,
        "stem": selected["stem"],
        "image_path": selected["image_path"],
        "json_path": selected["json_path"],
    })

representative_df = pd.DataFrame(representative_rows)
display(representative_df)

,code,stem,image_path,json_path
0,B0,S63_DATA3_B0_L1_D2023-08-25-10-50_001_000254,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
1,B1,S63_DATA3_B1_L1_D2023-08-25-10-42_001_000182,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
2,H0,S63_DATA3_H0_L1_D2023-09-07-13-08_001_000085,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
3,H1,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000058,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
4,M0,S63_DATA3_M0_L1_D2023-08-25-16-49_001_000001,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
5,M1,S63_DATA3_M1_L1_D2023-10-18-14-01_022_000008,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...


In [10]:
structure_frames = []

for row in representative_rows:
    code = row["code"]
    json_path = Path(row["json_path"])

    data = load_json(json_path)
    structure = flatten_json_structure(data)

    structure_df = pd.DataFrame(structure)
    structure_df.insert(0, "code", code)
    structure_df.insert(1, "json_path", str(json_path))

    structure_frames.append(structure_df)

    print("=" * 100)
    print("PPE 코드:", code)
    print("JSON:", json_path)
    display(structure_df.head(150))

PPE 코드: B0
JSON: /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_안전대미착용-B0/S63_DATA3_B0_L1_D2023-08-25-10-50_001_000254.json


,code,json_path,path,type,example
0,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root,dict,5 keys
1,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info,dict,2 keys
2,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info.name,str,37.S63-D3-B0
3,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info.id,str,10
4,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories,list,length=1
5,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0],dict,2 keys
6,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].name,str,안전장비
7,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].value,list,length=1
8,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].value[0],str,안전대 미착용
9,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.class,list,length=1


PPE 코드: B1
JSON: /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_안전대착용-B1/S63_DATA3_B1_L1_D2023-08-25-10-42_001_000182.json


,code,json_path,path,type,example
0,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root,dict,5 keys
1,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info,dict,2 keys
2,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info.name,str,38.S63-D3-B1
3,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info.id,str,17
4,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories,list,length=1
5,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0],dict,2 keys
6,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].name,str,안전장비
7,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].value,list,length=1
8,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].value[0],str,안전대 착용
9,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.class,list,length=1


PPE 코드: H0
JSON: /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_안전모미착용-H0/S63_DATA3_H0_L1_D2023-09-07-13-08_001_000085.json


...

,code,json_path,path,type,example
0,H1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root,dict,5 keys
1,H1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info,dict,2 keys
2,H1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info.name,str,36.S63-D3-H1
3,H1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info.id,str,16
4,H1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories,list,length=1
5,H1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0],dict,2 keys
6,H1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].name,str,안전장비
7,H1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].value,list,length=1
8,H1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].value[0],str,안전모 착용
9,H1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.class,list,length=1


PPE 코드: M0
JSON: /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_용접마스크미착용-M0/S63_DATA3_M0_L1_D2023-08-25-16-49_001_000001.json


,code,json_path,path,type,example
0,M0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root,dict,5 keys
1,M0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info,dict,2 keys
2,M0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info.name,str,39.S63-D3-M0
3,M0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info.id,str,18
4,M0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories,list,length=1
5,M0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0],dict,2 keys
6,M0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].name,str,안전장비
7,M0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].value,list,length=1
8,M0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].value[0],str,용접마스크 미착용
9,M0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.class,list,length=1


PPE 코드: M1
JSON: /content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_안전장비-S63_DATA3_용접마스크착용-M1/S63_DATA3_M1_L1_D2023-10-18-14-01_022_000008.json


,code,json_path,path,type,example
0,M1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root,dict,5 keys
1,M1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info,dict,2 keys
2,M1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info.name,str,40.S63-D3-M1
3,M1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.data_info.id,str,19
4,M1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories,list,length=1
5,M1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0],dict,2 keys
6,M1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].name,str,안전장비
7,M1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].value,list,length=1
8,M1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.categories[0].value[0],str,용접마스크 착용
9,M1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,root.class,list,length=1


In [11]:
from tqdm.auto import tqdm


def collect_key_paths(
    obj,
    prefix="root",
    paths=None,
    max_depth=8,
    current_depth=0,
):
    if paths is None:
        paths = set()

    if current_depth > max_depth:
        return paths

    if isinstance(obj, dict):
        for key, value in obj.items():
            child_path = f"{prefix}.{key}"
            paths.add(child_path)

            collect_key_paths(
                value,
                prefix=child_path,
                paths=paths,
                max_depth=max_depth,
                current_depth=current_depth + 1,
            )

    elif isinstance(obj, list):
        list_path = f"{prefix}[]"
        paths.add(list_path)

        # 구조 확인 목적이므로 첫 요소 중심
        for value in obj[:1]:
            collect_key_paths(
                value,
                prefix=list_path,
                paths=paths,
                max_depth=max_depth,
                current_depth=current_depth + 1,
            )

    return paths

In [12]:
KEY_SCAN_LIMIT = min(2000, len(pairs_df))

# 코드별·폴더별 편향을 줄이기 위해 균등 간격 샘플
if KEY_SCAN_LIMIT > 0:
    sample_indices = (
        pd.Series(range(len(pairs_df)))
        .sample(n=KEY_SCAN_LIMIT, random_state=42)
        .sort_values()
        .tolist()
    )
else:
    sample_indices = []

key_path_counter = Counter()
json_load_errors = []

for idx in tqdm(sample_indices, desc="JSON 키 구조 검사"):
    json_path = Path(pairs_df.iloc[idx]["json_path"])

    try:
        data = load_json(json_path)
        key_paths = collect_key_paths(data)

        for key_path in key_paths:
            key_path_counter[key_path] += 1

    except Exception as e:
        json_load_errors.append({
            "json_path": str(json_path),
            "error": repr(e),
        })

key_paths_df = pd.DataFrame(
    [
        {
            "key_path": key_path,
            "file_count": count,
            "ratio": count / max(KEY_SCAN_LIMIT, 1),
        }
        for key_path, count in key_path_counter.most_common()
    ]
)

print("검사 JSON 수:", KEY_SCAN_LIMIT)
print("JSON 로드 오류:", len(json_load_errors))

display(key_paths_df.head(200))

JSON 키 구조 검사:   0%|          | 0/2000 [00:00<?, ?it/s]

검사 JSON 수: 2000
JSON 로드 오류: 0


,key_path,file_count,ratio
0,root.images.device,2000,1.000
1,root.categories[].value,2000,1.000
2,root.class[].name,2000,1.000
3,root.images.environment,2000,1.000
4,root.categories[],2000,1.000
5,root.data_info.id,2000,1.000
6,root.class[].id,2000,1.000
7,root.annotations,2000,1.000
8,root.annotations[],2000,1.000
9,root.images.height,2000,1.000


In [13]:
CANDIDATE_KEYWORDS = [
    "annotation",
    "annotations",
    "polygon",
    "segmentation",
    "point",
    "points",
    "bbox",
    "box",
    "class",
    "category",
    "label",
    "code",
    "person",
    "worker",
    "image",
    "file_name",
    "filename",
    "width",
    "height",
]

In [14]:
candidate_mask = key_paths_df["key_path"].str.lower().apply(
    lambda path: any(keyword in path for keyword in CANDIDATE_KEYWORDS)
)

candidate_keys_df = key_paths_df[candidate_mask].copy()

display(candidate_keys_df.head(300))

,key_path,file_count,ratio
0,root.images.device,2000,1.000
2,root.class[].name,2000,1.000
3,root.images.environment,2000,1.000
6,root.class[].id,2000,1.000
7,root.annotations,2000,1.000
8,root.annotations[],2000,1.000
9,root.images.height,2000,1.000
10,root.images.file_name,2000,1.000
12,root.images,2000,1.000
13,root.images.angle,2000,1.000


In [15]:
for row in representative_rows:
    code = row["code"]
    json_path = Path(row["json_path"])

    data = load_json(json_path)
    formatted = json.dumps(
        data,
        ensure_ascii=False,
        indent=2,
    )

    print("\n" + "=" * 120)
    print(f"[{code}] {json_path.name}")
    print("=" * 120)
    print(formatted[:15000])

    if len(formatted) > 15000:
        print("\n... 출력 생략 ...")


[B0] S63_DATA3_B0_L1_D2023-08-25-10-50_001_000254.json
{
  "data_info": {
    "name": "37.S63-D3-B0",
    "id": "10"
  },
  "categories": [
    {
      "name": "안전장비",
      "value": [
        "안전대 미착용"
      ]
    }
  ],
  "class": [
    {
      "id": 0,
      "name": "안전대 미착용몸통",
      "keypoints": [
        ""
      ]
    }
  ],
  "images": {
    "id": 1164269,
    "file_name": "S63_DATA3_B0_L1_D2023-08-25-10-50_001_000254.jpg",
    "width": 1920,
    "height": 1080,
    "date": "2023-08-25 10:50:00",
    "location": "한화오션에코텍",
    "environment": "실내",
    "device": "고프로",
    "angle": 0
  },
  "annotations": [
    {
      "object_id": 1,
      "object_class": 0,
      "bbox": [
        1404.7182283658417,
        190.459518599562,
        1905.1387459156142,
        878.0743982494512
      ],
      "polygon": [
        1397.9951553791,
        268.1677787683,
        1474.5365642813,
        238.3391414755,
        1542.0731015479,
        211.8873310461,
        1601.1675716563,


In [16]:
def find_lists(obj, prefix="root", result=None):
    if result is None:
        result = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            find_lists(
                value,
                prefix=f"{prefix}.{key}",
                result=result,
            )

    elif isinstance(obj, list):
        item_types = Counter(type(item).__name__ for item in obj)

        sample_keys = None
        if obj and isinstance(obj[0], dict):
            sample_keys = sorted(obj[0].keys())

        result.append({
            "path": prefix,
            "length": len(obj),
            "item_types": dict(item_types),
            "first_item_keys": sample_keys,
        })

        for index, item in enumerate(obj[:2]):
            find_lists(
                item,
                prefix=f"{prefix}[{index}]",
                result=result,
            )

    return result

In [17]:
list_analysis_rows = []

for row in representative_rows:
    code = row["code"]
    json_path = Path(row["json_path"])
    data = load_json(json_path)

    list_rows = find_lists(data)

    for list_row in list_rows:
        list_row["code"] = code
        list_row["json_path"] = str(json_path)
        list_analysis_rows.append(list_row)

list_analysis_df = pd.DataFrame(list_analysis_rows)

columns = [
    "code",
    "path",
    "length",
    "item_types",
    "first_item_keys",
    "json_path",
]

display(
    list_analysis_df[columns]
    .sort_values(["code", "length"], ascending=[True, False])
    .head(200)
)

,code,path,length,item_types,first_item_keys,json_path
6,B0,root.annotations[0].polygon,72,{'float': 72},None,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
5,B0,root.annotations[0].bbox,4,{'float': 4},None,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
0,B0,root.categories,1,{'dict': 1},"[name, value]",/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
1,B0,root.categories[0].value,1,{'str': 1},None,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
2,B0,root.class,1,{'dict': 1},"[id, keypoints, name]",/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
3,B0,root.class[0].keypoints,1,{'str': 1},None,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
4,B0,root.annotations,1,{'dict': 1},"[bbox, categories, keypoint, object_class, obj...",/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
8,B0,root.annotations[0].categories,1,{'dict': 1},"[name, value]",/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
7,B0,root.annotations[0].keypoint,0,{},None,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
19,B1,root.annotations[1].polygon,90,{'float': 90},None,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...


In [18]:
def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def describe_numeric_list(value):
    if not isinstance(value, list):
        return None

    if not value:
        return {
            "structure": "empty_list",
            "length": 0,
        }

    # [x1, y1, x2, y2, ...]
    if all(is_number(item) for item in value):
        return {
            "structure": "flat_numeric_list",
            "length": len(value),
            "sample": value[:10],
        }

    # [[x1, y1], [x2, y2], ...]
    if all(
        isinstance(item, list)
        and len(item) >= 2
        and is_number(item[0])
        and is_number(item[1])
        for item in value
    ):
        return {
            "structure": "point_pair_list",
            "length": len(value),
            "sample": value[:5],
        }

    return None


def find_numeric_lists(obj, prefix="root", result=None):
    if result is None:
        result = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            description = describe_numeric_list(value)

            if description is not None:
                result.append({
                    "path": f"{prefix}.{key}",
                    **description,
                })

            find_numeric_lists(
                value,
                prefix=f"{prefix}.{key}",
                result=result,
            )

    elif isinstance(obj, list):
        for index, value in enumerate(obj[:20]):
            find_numeric_lists(
                value,
                prefix=f"{prefix}[{index}]",
                result=result,
            )

    return result

In [19]:
numeric_list_rows = []

for row in representative_rows:
    code = row["code"]
    json_path = Path(row["json_path"])
    data = load_json(json_path)

    findings = find_numeric_lists(data)

    for finding in findings:
        finding["code"] = code
        finding["json_path"] = str(json_path)
        numeric_list_rows.append(finding)

numeric_lists_df = pd.DataFrame(numeric_list_rows)

display(numeric_lists_df.head(300))

,path,structure,length,sample,code,json_path
0,root.annotations[0].bbox,flat_numeric_list,4,"[1404.7182283658417, 190.459518599562, 1905.13...",B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
1,root.annotations[0].polygon,flat_numeric_list,72,"[1397.9951553791, 268.1677787683, 1474.5365642...",B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
2,root.annotations[0].keypoint,empty_list,0,NaN,B0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
3,root.annotations[0].bbox,flat_numeric_list,4,"[42.3700000000001, 455.9399999999998, 195.5244...",B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
4,root.annotations[0].polygon,empty_list,0,NaN,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
5,root.annotations[0].keypoint,empty_list,0,NaN,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
6,root.annotations[1].bbox,empty_list,0,NaN,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
7,root.annotations[1].polygon,flat_numeric_list,90,"[93.776401444, 455.9448121032, 79.7879145282, ...",B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
8,root.annotations[1].keypoint,empty_list,0,NaN,B1,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
9,root.annotations[0].bbox,flat_numeric_list,4,"[1647.3985223727282, 462.6999999999986, 1756.6...",H0,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...


In [20]:
from PIL import Image


image_size_rows = []

for row in representative_rows:
    image_path = Path(row["image_path"])

    with Image.open(image_path) as image:
        width, height = image.size

    image_size_rows.append({
        "code": row["code"],
        "stem": row["stem"],
        "actual_width": width,
        "actual_height": height,
        "image_path": str(image_path),
        "json_path": row["json_path"],
    })

image_sizes_df = pd.DataFrame(image_size_rows)
display(image_sizes_df)

,code,stem,actual_width,actual_height,image_path,json_path
0,B0,S63_DATA3_B0_L1_D2023-08-25-10-50_001_000254,1920,1080,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
1,B1,S63_DATA3_B1_L1_D2023-08-25-10-42_001_000182,1920,1080,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
2,H0,S63_DATA3_H0_L1_D2023-09-07-13-08_001_000085,1920,1080,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
3,H1,S63_DATA3_H1_L1_D2023-09-14-10-23_001_000058,1920,1080,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
4,M0,S63_DATA3_M0_L1_D2023-08-25-16-49_001_000001,1920,1080,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...
5,M1,S63_DATA3_M1_L1_D2023-10-18-14-01_022_000008,1920,1080,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...,/content/drive/MyDrive/빅프/081_스마트야드_압축해제/3.개방데...


In [21]:
pairs_output_path = REPORT_ROOT / "image_json_pairs.csv"
representative_output_path = REPORT_ROOT / "representative_json_files.csv"
key_paths_output_path = REPORT_ROOT / "json_key_paths.csv"
candidate_keys_output_path = REPORT_ROOT / "candidate_annotation_keys.csv"
list_analysis_output_path = REPORT_ROOT / "json_list_analysis.csv"
numeric_lists_output_path = REPORT_ROOT / "numeric_coordinate_candidates.csv"
json_errors_output_path = REPORT_ROOT / "json_load_errors.csv"

pairs_df.to_csv(
    pairs_output_path,
    index=False,
    encoding="utf-8-sig",
)

representative_df.to_csv(
    representative_output_path,
    index=False,
    encoding="utf-8-sig",
)

key_paths_df.to_csv(
    key_paths_output_path,
    index=False,
    encoding="utf-8-sig",
)

candidate_keys_df.to_csv(
    candidate_keys_output_path,
    index=False,
    encoding="utf-8-sig",
)

list_analysis_df.to_csv(
    list_analysis_output_path,
    index=False,
    encoding="utf-8-sig",
)

numeric_lists_df.to_csv(
    numeric_lists_output_path,
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(json_load_errors).to_csv(
    json_errors_output_path,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료")
print("pairs:", pairs_output_path)
print("representative:", representative_output_path)
print("key paths:", key_paths_output_path)
print("candidate keys:", candidate_keys_output_path)
print("list analysis:", list_analysis_output_path)
print("numeric lists:", numeric_lists_output_path)
print("errors:", json_errors_output_path)

저장 완료
pairs: /content/drive/MyDrive/빅프/shared_backbone_workspace/reports/image_json_pairs.csv
representative: /content/drive/MyDrive/빅프/shared_backbone_workspace/reports/representative_json_files.csv
key paths: /content/drive/MyDrive/빅프/shared_backbone_workspace/reports/json_key_paths.csv
candidate keys: /content/drive/MyDrive/빅프/shared_backbone_workspace/reports/candidate_annotation_keys.csv
list analysis: /content/drive/MyDrive/빅프/shared_backbone_workspace/reports/json_list_analysis.csv
numeric lists: /content/drive/MyDrive/빅프/shared_backbone_workspace/reports/numeric_coordinate_candidates.csv
errors: /content/drive/MyDrive/빅프/shared_backbone_workspace/reports/json_load_errors.csv


In [22]:
print("=" * 80)
print("1. 파일 매칭")
print("=" * 80)
print(f"이미지: {len(image_paths):,}")
print(f"JSON: {len(json_paths):,}")
print(f"매칭: {len(pairs_df):,}")
print(f"JSON 없는 이미지: {len(images_without_json):,}")
print(f"이미지 없는 JSON: {len(jsons_without_image):,}")

print("\n" + "=" * 80)
print("2. PPE 코드")
print("=" * 80)
print(
    pairs_df["filename_ppe_code"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "=" * 80)
print("3. annotation 관련 후보 키")
print("=" * 80)
print(
    candidate_keys_df[
        candidate_keys_df["key_path"].str.lower().str.contains(
            "annot|polygon|segment|bbox|class|category|label|point",
            regex=True,
        )
    ]
    .head(100)
    .to_string(index=False)
)

print("\n" + "=" * 80)
print("4. 가장 큰 list 후보")
print("=" * 80)

if not list_analysis_df.empty:
    print(
        list_analysis_df[
            ["code", "path", "length", "first_item_keys"]
        ]
        .sort_values("length", ascending=False)
        .head(30)
        .to_string(index=False)
    )

print("\n" + "=" * 80)
print("5. 좌표 배열 후보")
print("=" * 80)

if not numeric_lists_df.empty:
    print(
        numeric_lists_df[
            ["code", "path", "structure", "length", "sample"]
        ]
        .head(50)
        .to_string(index=False)
    )

1. 파일 매칭
이미지: 76,966
JSON: 76,966
매칭: 76,966
JSON 없는 이미지: 0
이미지 없는 JSON: 0

2. PPE 코드
filename_ppe_code
B0    15336
H1    15254
H0    15190
B1    14966
M0     8415
M1     7805

3. annotation 관련 후보 키
                             key_path  file_count  ratio
                    root.class[].name        2000  1.000
                      root.class[].id        2000  1.000
                     root.annotations        2000  1.000
                   root.annotations[]        2000  1.000
                           root.class        2000  1.000
             root.class[].keypoints[]        2000  1.000
               root.class[].keypoints        2000  1.000
                         root.class[]        2000  1.000
              root.annotations[].bbox        1998  0.999
        root.annotations[].keypoint[]        1998  0.999
      root.annotations[].categories[]        1998  0.999
            root.annotations[].bbox[]        1998  0.999
 root.annotations[].categories[].name        1998  0.999
   